# User-written vector reaction-diffusion equations

Two vector components share a genuine user-defined UFL operator. This is a
coercive reaction-diffusion problem, with no pressure or incompressibility
constraint. On the unit square, set $\alpha=2$ and
$u=(1+x+2y,-1+3x-y)$. Its source is independently $f=\alpha u$ because
the affine field has zero Laplacian. The vector normal flux is
$\lambda=-\nabla u\,n$ using each fixed macroface normal.

$$
\begin{aligned}
a_T(u,v)&=(\nabla u,\nabla v)_T+\alpha(u,v)_T,\\
L_T(v)&=(f,v)_T,\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tu&=-\langle u,\mu\rangle_{\partial T}.
\end{aligned}
$$

P1 vector coefficients and P0 vector traces are component-interleaved.
Positive reaction removes the local constant kernel, so no retained mode or
gauge is needed. The global right-hand side is explicitly
$-\langle u_D,\mu\rangle_{\partial\Omega}$; $D_T=g_T=0$.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/6cb450f89995c92dc2582478964a72ce75312349db2cb9a85d422c8b7281ecad/vector_ufl-companion.zip"
COMPANION_SHA256 = "6cb450f89995c92dc2582478964a72ce75312349db2cb9a85d422c8b7281ecad"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("foundations/operators/vector_ufl.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)

import numpy as np


In [ ]:
from scipy import sparse
from pymhm import Equation, LocalEquations, MultiscaleProblem, assemble, columns, rows
from pymhm.meshes.triangle import TriangleMesh
from pymhm.fem.traces.interval import SkeletonSpace
from pymhm.fem.scalar.operators import p1_operators, face_integration, boundary_data
from examples.variational_darcy import NativeFormProvider

alpha = 2.0
gradient = np.array([[1.0, 2.0], [3.0, -1.0]])
offset = np.array([1.0, -1.0])

def exact_vector(points):
    """Evaluate the independently prescribed affine vector field."""
    return points @ gradient.T + offset

mesh = TriangleMesh.unit_square()
skeleton = SkeletonSpace(mesh, components=2)
scalar_skeleton = SkeletonSpace(mesh)

def portable_vector_forms(cell):
    """Declare the nodal component blocks without selecting a physics solver."""
    fine = mesh.submesh(cell, 2)
    a, _, _ = p1_operators(fine, reaction=alpha)
    loads = []
    for component in range(2):
        def component_source(points, component=component):
            """Return one component of the independently differentiated source."""
            return alpha * exact_vector(points)[:, component]
        _, _, load = p1_operators(fine, source=component_source)
        loads.append(load)
    b, _ = face_integration(mesh, cell, fine, scalar_skeleton)
    vector_b = sparse.kron(sparse.csc_matrix(b), np.eye(2), format="csc").toarray()
    return LocalEquations(
        sparse.kron(a, np.eye(2), format="csc"), np.stack(loads, axis=1).ravel(),
        columns(*vector_b.T), rows(*(-vector_b.T)), skeleton.cell_dofs(cell),
        metadata={"points": fine.points.copy()},
    )

boundary, _ = boundary_data(skeleton, exact_vector)
global_equation = Equation(0, -boundary)
problem = MultiscaleProblem(global_equation, portable_vector_forms,
                            range(len(mesh.cells)), skeleton.size, (0,) * len(mesh.cells))
system = assemble(problem)
solution = system.solve()
errors = [float(np.max(np.abs(field.reshape(-1, 2) - exact_vector(record["points"]))))
          for field, record in zip(solution.fields, system.local_metadata, strict=True)]
expected_trace = np.concatenate([-gradient @ normal for normal in mesh.normals])
np.testing.assert_allclose(solution.trace, expected_trace, atol=2e-12, rtol=0)
assert max(errors) < 1e-10
print({"assembly": "Basix coefficient forms", "vector_nodal_error_by_cell": errors,
       "global_original_relative_residual": solution.raw_residual})


## Supply the same operator as native UFL

The mesh context is created inside the native provider. The application creates its own vector function space and writes every volume and boundary form. The generic compiler accepts the C rows with their trial argument. Native execution uses COMM_SELF and returns only numerical response data. The optional cell records absence of DOLFINx explicitly; native validation requires compatible DOLFINx/UFL dependencies.


In [ ]:
import importlib.util

native_available = importlib.util.find_spec("dolfinx") is not None
if native_available:
    import ufl
    from dolfinx import fem

    def user_vector_forms(context):
        """Write vector UFL forms directly on the worker's local mesh."""
        space = fem.functionspace(context.space.mesh, ("Lagrange", 1, (2,)))
        u, v = ufl.TrialFunction(space), ufl.TestFunction(space)
        x = ufl.SpatialCoordinate(context.space.mesh)
        exact = ufl.as_vector((1 + x[0] + 2 * x[1], -1 + 3 * x[0] - x[1]))
        dx = ufl.dx(domain=context.space.mesh)
        return LocalEquations(
            (ufl.inner(ufl.grad(u), ufl.grad(v)) + alpha * ufl.inner(u, v)) * dx,
            ufl.inner(alpha * exact, v) * dx,
            columns(*(float(sign) * v[component] * context.ds(side + 1)
                      for side, sign in enumerate(context.signs) for component in range(2))),
            rows(*(-float(sign) * u[component] * context.ds(side + 1)
                   for side, sign in enumerate(context.signs) for component in range(2))),
            context.trace_dofs,
        )

    provider = NativeFormProvider(mesh, skeleton, user_vector_forms, 2)
    native_problem = MultiscaleProblem(global_equation, provider, range(len(mesh.cells)),
                                      skeleton.size, (0,) * len(mesh.cells))
    native_system = assemble(native_problem)
    native_solution = native_system.solve()
    native_errors = [
        float(np.max(np.abs(field.reshape(-1, 2) - exact_vector(record["points"]))))
        for field, record in zip(native_solution.fields, native_system.local_metadata, strict=True)
    ]
    np.testing.assert_allclose(native_solution.trace, solution.trace, atol=2e-12, rtol=0)
    assert max(native_errors) < 1e-10
    print({"native_available": True, "assembly": "UFL/DOLFINx user forms",
           "vector_nodal_error_by_cell": native_errors,
           "global_original_relative_residual": native_solution.raw_residual})
else:
    print({"native_available": False,
           "status": "Native vector forms not executed: select the Pixi fem kernel."})
